# Data Cleaning – MoMo Top-up Case Study

The objective of this notebook is to clean and prepare the three raw datasets from `momo_top_up.xlsx`:
1. **Transactions**: Transaction records.
2. **User Info**: Demographic data of users.
3. **Commission**: Commission rates data.

## 1. Setup & Initialization

In [4]:
import pandas as pd
from pathlib import Path
import os

In [5]:
BASE_DIR = Path.cwd().parent
SRC_PATH   = BASE_DIR / "Data" / "raw" / "momo_top_up.xlsx"
OUT_DIR   = BASE_DIR / "Data" / "processed"

## 2. Transactions Data (`df_transactions`)
Load the transactions dataset from the first sheet and perform an initial overview, including checking for duplicated `order_id`s.

In [6]:
df_transactions = pd.read_excel(SRC_PATH, sheet_name=0)
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13495 entries, 0 to 13494
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   user_id          13495 non-null  int64 
 1   order_id         13495 non-null  int64 
 2   Date             13495 non-null  object
 3   Amount           13495 non-null  object
 4   Merchant_id      13495 non-null  int64 
 5   Purchase_status  2235 non-null   object
dtypes: int64(3), object(3)
memory usage: 632.7+ KB


### Preview dataframe Transactions

In [7]:
df_transactions.head()

,user_id,order_id,Date,Amount,Merchant_id,Purchase_status
0,21269588,4169517626,2020-01-01,"10,000",13,NaN
1,28097592,4170276686,2020-01-01,"20,000",13,NaN
2,47435144,4166729310,2020-01-01,"10,000",12,NaN
3,29080935,4174460303,2020-01-01,"10,000",13,NaN
4,14591075,4168216749,2020-01-01,"10,000",12,NaN


### Clean `Date` Column

In [8]:
a = df_transactions['Date']

mask = pd.to_datetime(a, errors='coerce').isna()

a[mask].unique()

array(['27/9/2020', '28/9/2020', '29/9/2020', '30/9/2020'], dtype=object)

The date formats in the raw data are inconsistent (mixed between `DD/MM/YYYY` and `YYYY-MM-DD`). 
And some values aren't right about the logic

In [9]:
df_transactions['Date'] = pd.to_datetime(df_transactions['Date'],
    format = "mixed",
    dayfirst= True,
    errors = "coerce")

#### Checking if there are any null values left

In [10]:
df_transactions['Date'] = pd.to_datetime(df_transactions['Date'],format='mixed',errors= 'coerce',dayfirst= True)
print(df_transactions['Date'].isnull().sum())

0


### Checking duplicated rows

In [11]:
print(df_transactions.duplicated('order_id').sum())

0


### Clean Amount column
- **Amount**: Delete "," , change data type into (float)

In [12]:
df_transactions['Amount'].value_counts()

Amount
20,000       3357
10,000       3245
50,000       3105
100,000      2228
30,000        830
200,000       468
300,000       110
500,000       100
40,000         31
60,000          8
1,000,000       5
400,000         3
2,500,000       2
294,234         1
2,000,000       1
250,000         1
Name: count, dtype: int64

There are 1 values that outlier are 294.234, so i will drop it

In [13]:
df_transactions['Amount'] = pd.to_numeric(
        df_transactions['Amount'].str.replace(',', '', regex=False),
        errors='coerce'
    )

In [14]:
print(df_transactions['Amount'].isna().sum())

0


In [15]:
df_transactions = df_transactions[df_transactions['Amount'] != 294234]

In [16]:
df_transactions['Amount'].value_counts()

Amount
20000      3357
10000      3245
50000      3105
100000     2228
30000       830
200000      468
300000      110
500000      100
40000        31
60000         8
1000000       5
400000        3
2500000       2
2000000       1
250000        1
Name: count, dtype: int64

### Clean Purchase_status column


In [17]:
df_transactions['Purchase_status'] = (df_transactions['Purchase_status'] == 'Mua hộ').astype(int)

## 3. User Info Data 
Loading the demographic data from the third sheet of the Excel file.

In [18]:
df_user_info = pd.read_excel(SRC_PATH, sheet_name=2)
df_user_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13428 entries, 0 to 13427
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   User_id          13428 non-null  int64 
 1   First_tran_date  13428 non-null  object
 2   Location         13428 non-null  object
 3   Age              13428 non-null  object
 4   Gender           13428 non-null  object
dtypes: int64(1), object(4)
memory usage: 524.7+ KB


#### Rename columns User_id into user_id to match with Transactions

In [19]:
df_user_info.rename(columns={'User_id': 'user_id'}, inplace=True)

### Handle duplicates Users
There are some "user_id" that is duplicated, we will sort them by 'First_tran_date' and keep the latest one for each user.

In [20]:
print(df_user_info.duplicated('user_id').sum())

38


In [21]:
df_user_info = (
    df_user_info.sort_values('First_tran_date')
    .drop_duplicates('user_id',keep='last')
)

### Clean 'First_tran_date'
Firstly , I will check some values that made the convert error by using the function 'unique()'. It's show that some years contain typos (ex: `9917` instead of `2017`, `3020` instead of `2020`)

In [22]:
s = df_user_info['First_tran_date']

mask = pd.to_datetime(s, errors='coerce').isna()

s[mask].unique()

array(['3020-05-28', '3020-12-01', '9917-09-22', '9917-09-23',
       '9917-09-25', '9917-09-27', '9917-09-28', '9917-10-01',
       '9917-10-04', '9917-10-05', '9917-10-06', '9917-10-07',
       '9917-10-09', '9917-10-10', '9917-10-11', '9917-10-20',
       '9917-10-22', '9917-10-26', '9917-10-31', '9917-11-01',
       '9917-11-10', '9917-12-01', '9917-12-20', '9917-12-27',
       '9918-01-01', '9918-02-16', '9918-02-28', '9918-04-16',
       '9918-06-06', '9918-07-20', '9918-09-27', '9918-10-01',
       '9918-10-25', '9918-11-15', '9918-12-29', '9919-02-09',
       '9919-03-17', '9919-03-19', '9919-03-22', '9919-04-07',
       '9919-05-21', '9919-07-02', '9919-09-02', '9919-09-14',
       '9919-09-15', '9919-10-06', '9919-11-06', '9919-11-18',
       '9919-11-20', '9919-12-14', '9920-01-03', '9920-02-14',
       '9920-03-04', '9920-03-05', '9920-03-10', '9920-03-12',
       '9920-03-24', '9920-04-27', '9920-05-31', '9920-12-06',
       '9920-12-12', '9920-12-16'], dtype=object)

The data isn't right about the logic, it's still have a pattern( for ex : 99,30 is 20), so I use mapping to correct them

In [23]:
year_mapping = {
    '9917': '2017',
    '9918': '2018',
    '9919': '2019',
    '9920': '2020',
    '3020': '2020'
}

df_user_info['First_tran_date'] = (
    df_user_info['First_tran_date']
    .str[:4]
    .replace(year_mapping)
    + df_user_info['First_tran_date'].str[4:]
)

df_user_info['First_tran_date'] = pd.to_datetime(
    df_user_info['First_tran_date'], errors='coerce'
)

In [24]:
print(df_user_info['First_tran_date'].isnull().sum())


0


### Standardize `Location`
Group and clean varying location strings into standardized categories (e.g., convert `Ho Chi Minh City` to `HCMC`, and group `Unknown` and `Other` into `Other Cities`).

In [25]:
print(df_user_info['Location'].value_counts(dropna=False).sort_index())


Location
HCMC                4100
HN                  1434
Ho Chi Minh City      62
Other               1030
Other Cities        6004
Unknown              760
Name: count, dtype: int64


In [26]:
mapping_location = {
    'Ho Chi Minh City': 'HCMC',   
    'Other': 'Other Cities',      
    'Unknown': 'Other Cities'     
}

df_user_info['Location'] = df_user_info['Location'].replace(mapping_location)

print(df_user_info['Location'].value_counts())

Location
Other Cities    7794
HCMC            4162
HN              1434
Name: count, dtype: int64


### Standardize `Gender`
Normalize gender values by converting everything to uppercase and mapping variations like `NỮ`, `F` to `FEMALE`, and `NAM`, `M` to `MALE`.

In [27]:
print(df_user_info['Gender'].value_counts(dropna=False).sort_index())

Gender
FEMALE    3394
M           92
MALE      6265
Nam       1368
Nữ        1244
f           55
female     901
male        71
Name: count, dtype: int64


In [28]:
df_user_info['Gender'] = df_user_info['Gender'].str.upper()

mapping_gender = {
    'NỮ': 'FEMALE',
    'F': 'FEMALE',
    'NAM': 'MALE',
    'M': 'MALE'
}
df_user_info['Gender'] = df_user_info['Gender'].replace(mapping_gender)
print(df_user_info['Gender'].value_counts())

Gender
MALE      7796
FEMALE    5594
Name: count, dtype: int64


## 4. Commission Data (`df_commission`)
Load the commission data from the first sheet.

In [29]:
df_commission = pd.read_excel(SRC_PATH, sheet_name=1)

## 5. Final Review & Export
Verify the structure of our final cleaned dataframes and export them into the `Data/processed/` directory. These cleaned `.csv` files will be used in the Exploratory Data Analysis (EDA) phase.

In [30]:
print(df_transactions.isnull().sum())
print(df_user_info.isnull().sum())


user_id            0
order_id           0
Date               0
Amount             0
Merchant_id        0
Purchase_status    0
dtype: int64
user_id            0
First_tran_date    0
Location           0
Age                0
Gender             0
dtype: int64


In [31]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

df_transactions.to_csv(OUT_DIR / "df_transactions.csv",index=False, encoding="utf-8-sig")
df_user_info.to_csv(OUT_DIR / "df_users.csv",index=False, encoding="utf-8-sig")
df_commission.to_csv(OUT_DIR / "df_commission.csv",index=False, encoding="utf-8-sig")